In [1]:
import pandas as pd
import numpy as np
import ollama
import re
import json

from src.io import read_tabular
from pathlib import Path
from tqdm.notebook import tqdm

from pydantic import BaseModel, Field
from typing import List, Optional, Literal

In [3]:
GEMMA = 'gemma4:31b-cloud'
ollama.pull(GEMMA)

client = ollama.Client()

In [5]:
codebook = {
    "type": {
        "definition": "Whether the target of a core sentence is an actor or an issue. Consists of two categories: 'actor-actor' or 'actor-issue'.",
        "clarification": "Political claims can be about other actors such as politicians, parties, movements, businesses, etc. If the target of a claim is another actor, it is an 'actor-actor' sentence. Political claims can also be about issues and policies without mentioning another actor. If the target of a claim is an issue, it is an 'actor-issue' sentence.",
        "negative_clarification": "Actor-actor sentences require there to be a second (organised) actor. Policies or plans are not actors but are coded as issues. Multiple actors being mentioned does not always imply an actor-actor relation.",
        "positive_example": "'Le Pen criticized Macron's lack of leadership' → 'actor-actor'",
        "negative_example": "'Starmer and Miliband propose a new plan to relieve Brits of high energy costs' → 'actor-issue'. While two people are mentioned, they are part of the same group (the Labour Party) and should be considered as one actor."
    },
    "subject": {
        "definition": "The subject of a core sentence is an actor who makes a political claim in the grammatical sentence.",
        "clarification": "A subject needs to be a nationally relevant political actor, such as a party, movement, or a civil society actor. Businesses, trade unions, experts, etc., can be actors too if their actions in the sentence relate directly to national politics. The subject is not necessarily equal to the grammatical subject.",
        "negative_clarification": "We do not code subjects where they are foreign actors. For example, Donald Trump pulling out of the Paris Agreement would not be coded as we are not interested in US politics. We do not code non-political actors unless they are making political demands or comment on the actions of political actors.",
        "positive_example": "'Friedrich Merz wants to reduce subsidies for renewable energies' → 'Friedrich Merz'",
        "negative_example": "'Volkswagen announced their intentions to phase out production of combustion engine cars by 2040.' The actor 'Volkswagen' is not a political actor, nor is it making a political demand. We do not code it as a subject."
    },
    "subject_organisation": {
        "definition": "The organisation (political party, institution, movement, etc.) the actor making a claim is part of or related to.",
        "clarification": "Organisations do not need to be actively mentioned to be coded. Use context and background knowledge to determine the organisation. Many actors are part of one or more organisations (political parties, institutions, movements, etc.). In this case, prioritise parties or movements, or use the organisation mentioned in the text.",
        "negative_clarification": "Do not code past organisational affiliations. For politicians, use party affiliations rather than institutional affiliations wherever possible.",
        "positive_example": "'Le Pen called Macron's climate plan the destruction of the French economy' → 'Rassemblement National'",
        "negative_example": "'Chancellor Merz announced his plans to propose a climate fund at the upcoming EU summit'. As we have an individual with a clear party affiliation, the correct code here is not 'German Government', but 'CDU/CSU'."
    },
    "direction": {
        "definition": "The position (coded as 'support', 'opposition', or 'ambiguous') that the subject takes towards the issue or object of the core sentence.",
        "clarification": "In an actor-actor sentence, the direction refers to how the subject positions itself towards the object. In an actor-issue sentence, the direction refers to how the subject positions itself towards the issue referenced in the sentence. Try to assign a clear direction (i.e., 'support' or 'opposition') whenever possible.",
        "negative_clarification": "In an actor-actor sentence, disregard the issue when coding direction. Do not use background knowledge for this variable - only code the direction as it appears in the specific sentence in the text.",
        "positive_example": "'Extinction Rebellion protested against the government's plan' → 'opposition'",
        "negative_example": "'The Greens voted against the proposed law to subsidise renewables' → The text mentions opposition of the Greens ('voted against'). Code as the Greens opposing renewable subsidies, even if you know they generally support subsidies for renewables."
    },
    "object": {
        "definition": "The object of a core sentence is an organised actor (party, movement, business, etc.) that is the target of a political claim.",
        "clarification": "Objects do not necessarily have to be nationally relevant actors. Any organisation or actor that is talked about by a relevant subject is a valid object to be coded, even if it is non-political (e.g., businesses) or from another country.",
        "negative_clarification": "Do not code an object in an actor-issue sentence. Objects can only be actors (i.e., people or organisations). Do not code abstract or vague actors (e.g., 'politicians', 'the industry').",
        "positive_example": "'Fridays for Future's actions were criticised by PM Starmer in his parliamentary speech' → 'Fridays for Future'",
        "negative_example": "'Meloni also emphasised the importance of a secure gas supply for the Italian industry'. The target of this sentence is 'a secure gas supply', which is an issue rather than an actor. Do not code an object. 'the Italian industry' is too vague to be considered an object of a core sentence."
    },
    "object_organisation": {
        "definition": "The organisation (political party, institution, movement, etc.) the target of a claim is part of or related to.",
        "clarification": "Organisations do not need to be actively mentioned to be coded. Use context and background knowledge to determine the organisation. Many actors are part of one or more organisations (political parties, institutions, movements, etc.). In this case, prioritise parties or movements, or use the organisation mentioned in the text.",
        "negative_clarification": "Do not code past organisational affiliations. For politicians, use party affiliations rather than institutional affiliations wherever possible.",
        "positive_example": "'Extinction Rebellion blockaded and attacked a Shell office building' → 'Shell'. Even though Shell is not a political actor, it is the target of a political action.",
        "negative_example": "'Nigel Farage demands an end to the government's climate mania' → 'UK Government'. Use background knowledge to prioritise party affiliation, rather than coding institutions, where possible - the correct code would be 'Labour Party'."
    },
    "issue": {
        "definition": "An issue that is being referenced in a core sentence, either directly as the target of the sentence or in relation to the target.",
        "clarification": "In actor-issue sentences, the issue is the target of the core sentence, i.e., the thing that a claim is being made about. In actor-actor sentences, an issue can also be referenced as a justification or a reason for the statement. Code both these instances as issues. Issues always need to be coded as a position or action where 'supports' or 'opposes' has a clear meaning.",
        "negative_clarification": "Do not code issues that are simply mentioned without being connected to the claim. Do not code instances where an actor is simply mentioned in conjunction with an issue, but no claim is being made, such as a simple description of a phenomenon.",
        "positive_example": [
            "'The Romanian Prime Minister announced his plan to create a fund for relief for flood victims' → 'relieve flood victims'. This is the target of the core sentence.",
            "'Extinction Rebellion protested the government's decision to allow an extension of coal mining' → 'expanding coal mining'. The target of the sentence is 'the government', but coal mining is referenced as the reason for opposing the government."
        ],
        "negative_example": "'Meloni says she is preparing a relief package for high fuel costs.' → 'fuel costs'. It is unclear what this means. Use a clearer directional label such as 'relief for fuel costs' or 'lowering fuel costs'."
    }
}

In [6]:
class CoreSent(BaseModel):
    type: Literal['actor-actor', 'actor_issue', 'NA'] = Field(..., description = "type")
    subject: str = Field(..., description="subject")
    subject_organisation: str = Field(..., description="subject_organisation")
    direction: Literal["support", "opposition", "ambivalent", 'NA'] = Field(..., description = "direction")
    object: Optional[str] = Field(None, description = "object")
    object_organisation: Optional[str] = Field(None, description = "object_organisation")
    issue: Optional[str] = Field(None, description = "issue")

class CSResponse(BaseModel):
    sentence: str = Field(..., description="The grammatical sentence you coded")
    core_sents: Optional[List[CoreSent]] = Field(
        None,
        description="List of core sentences extracted from the sentence. Leave empty if none are detected."
    )

json_schema = CSResponse.model_json_schema()

In [11]:
sysprompt = f'''
You are an expert coder with training and expertise in analysing political claims. You follow British politics to the level of an interested, engaged daily news reader, and know the most important politicians, parties, and issues in the United Kingdom in late 2025/early 2026. Your task is to identify and code political claims relevant to British politics in newspaper articles using the following codebook:

{codebook}

## Input format
You are given up to five sentences published in a British newspaper, one of which is marked with > <. Code only the marked sentence, but use the other sentences to provide context to the marked sentence.

## Output format
Return a JSON using the following structure:
{json.dumps(json_schema)} 

Return nothing else.
'''

In [18]:
messages = [
    {"role": "system", "content":sysprompt},
    {"role": "user", "content": "Using the codebook you were provided, describe the correct value for subject for the following description: 'A claim is being made by Friedrich Merz about German politics.'"}
]

opts = {
    "seed": 42,
    "temperature": 0.0
}

chat = client.chat(
    model = GEMMA,
    messages = messages,
    options = opts
)

print(chat.message.content)

Friedrich Merz
